In [1]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("train.txt", sep= ';', header=None,names= ['text', 'emotions'])

In [3]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [5]:
emotions = df['emotions'].unique()

In [6]:
emo_numbers = {}
i = 0
for emo in emotions:
    emo_numbers[emo]= i
    i+=1
df['emotions'] = df['emotions'].map(emo_numbers)

In [7]:
df

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [8]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( df["text"], df["emotions"], test_size=0.20, random_state=42)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [13]:
feature_sets = {
    "Bag of Words": (X_train_bow, X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}



models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}


results = []

for feature_name, (X_train, X_test) in feature_sets.items():

    for model_name, model in models.items():

        
        model.fit(X_train, y_train)

        
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )


        results.append({
            "Feature Technique": feature_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })



results_df = pd.DataFrame(results)

results_df

C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Feature Technique,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.883125,0.882238,0.883125,0.881936
1,Bag of Words,Linear SVM,0.881875,0.881313,0.881875,0.881260
2,Bag of Words,Naive Bayes,0.739062,0.782701,0.739062,0.698183
3,TF-IDF,Logistic Regression,0.841250,0.847900,0.841250,0.833661
4,TF-IDF,Linear SVM,0.888437,0.887590,0.888437,0.887131
5,TF-IDF,Naive Bayes,0.617500,0.709283,0.617500,0.519369


In [14]:
print("BoW features:", len(vectorizer.get_feature_names_out()))
print("TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))

BoW features: 13501
TF-IDF features: 13501
